# Lab 08-06 · Latches, Flip-Flops and Sequential Timing

Companion to chapter [08-06](../../chapters/08-06-latches-and-flip-flops.md). Handout: [README.md](README.md).

The inverter here uses the square-law (level-1) model of the chapter: nMOS 0.65 µm and pMOS 1.0 µm wide, 0.15 µm long, $V_T$ = 0.45 V, $V_{DD}$ = 1.8 V. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

VDD, VT = 1.8, 0.45
bn, bp = 250e-6 * 0.65 / 0.15, 100e-6 * 1.0 / 0.15


def i_n(vin, vout):
    vov = vin - VT
    if vov <= 0:
        return 0.0
    return bn * (vov * vout - vout ** 2 / 2) if vout < vov else 0.5 * bn * vov ** 2


def i_p(vin, vout):
    vov, vsd = VDD - vin - VT, VDD - vout
    if vov <= 0:
        return 0.0
    return bp * (vov * vsd - vsd ** 2 / 2) if vsd < vov else 0.5 * bp * vov ** 2


def vtc(vin):
    # the output voltage at which pull-up and pull-down currents balance
    f = lambda vo: i_p(vin, vo) - i_n(vin, vo)
    lo, hi = 0.0, VDD
    if f(lo) <= 0:
        return 0.0
    if f(hi) >= 0:
        return VDD
    return brentq(f, lo, hi)


VM = brentq(lambda x: 0.5 * bn * (x - VT) ** 2 - 0.5 * bp * (VDD - x - VT) ** 2, VT, VDD - VT)
print(f"V_M = {VM:.4f} V")

V_M = 0.8456 V


## Part 1 · The butterfly

**🔮 Predict:** where do an inverter's transfer curve and its mirror image (the second inverter of the pair) cross?

In [2]:
vin = np.linspace(0, VDD, 721)
vout = np.array([vtc(x) for x in vin])
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(vin, vout, color="#0072B2", label="inverter 1: $V_2 = f(V_1)$")
ax.plot(vout, vin, color="#D55E00", label="inverter 2: $V_1 = f(V_2)$")
for p in [(0, VDD), (VDD, 0), (VM, VM)]:
    ax.plot(*p, "ko")
ax.annotate("metastable\n$V_M$", (VM, VM), xytext=(VM + 0.25, VM + 0.35), arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("$V_1$ (V)"); ax.set_ylabel("$V_2$ (V)"); ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.set_title("Cross-coupled inverters: three equilibria")
plt.show()

gain = np.gradient(vout, vin)
k = np.argmin(abs(vin - VM))
print(f"slope of the transfer curve near V_M: {gain[k]:.0f} (steep: the square-law model with lambda = 0 has no output conductance)")
# with lambda = 0 the curve is vertical at V_M: just below it the output is above V_M, just above it below
assert vtc(VM - 1e-4) > VM > vtc(VM + 1e-4)

slope of the transfer curve near V_M: -206 (steep: the square-law model with lambda = 0 has no output conductance)


## Part 2 · Resolving from near $V_M$ (the chapter's guided exercise)

**🔮 Predict:** starting 1 mV apart, how many time constants $\tau = C/g_m$ until the nodes are 0.9 V apart?

In [3]:
C = 10e-15
gm = bn * (VM - VT) + bp * (VDD - VM - VT)
tau = C / gm
pred = tau * math.log(0.9 / 1e-3)


def pair(t, y):
    v1, v2 = y
    return [(i_p(v2, v1) - i_n(v2, v1)) / C, (i_p(v1, v2) - i_n(v1, v2)) / C]


def apart(t, y):
    return abs(y[0] - y[1]) - 0.9


apart.terminal = True
sol = solve_ivp(pair, [0, 3e-10], [VM + 0.5e-3, VM - 0.5e-3], events=apart, max_step=1e-13, rtol=1e-9, atol=1e-12)
t_res = sol.t_events[0][0]
print(f"tau = {tau * 1e12:.2f} ps; ln(900) = {math.log(900):.2f} time constants; predicted {pred * 1e12:.1f} ps, simulated {t_res * 1e12:.1f} ps")
assert abs(t_res - pred) / pred < 0.03

full = solve_ivp(pair, [0, 2e-10], [VM + 0.5e-3, VM - 0.5e-3], max_step=1e-13, rtol=1e-9, atol=1e-12)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(full.t * 1e12, full.y[0], label="$V_1$"); ax[0].plot(full.t * 1e12, full.y[1], label="$V_2$")
ax[0].set_xlabel("time (ps)"); ax[0].set_ylabel("V"); ax[0].legend(); ax[0].set_title("node voltages")
ax[1].semilogy(full.t * 1e12, abs(full.y[0] - full.y[1]), label="simulated")
ax[1].semilogy(full.t * 1e12, 1e-3 * np.exp(full.t / tau), "--", label="$1\\,mV \\cdot e^{t/\\tau}$")
ax[1].set_ylim(5e-4, 3); ax[1].set_xlabel("time (ps)"); ax[1].set_title("difference, log scale"); ax[1].legend()
plt.show()

tau = 13.07 ps; ln(900) = 6.80 time constants; predicted 88.9 ps, simulated 89.9 ps


On the log plot the difference is a straight line, exponential growth at rate $1/\tau$, until the nodes approach the rails and the gain collapses. Halve the starting difference and you add $\tau \ln 2$ ≈ 9 ps; the time to resolve grows without bound as the start approaches $V_M$ exactly: that is metastability ([08-10](../../chapters/08-10-timing-metastability-cdc.md)).

## Part 3 · Latch versus flip-flop

In [4]:
def latch(clk, d, high=True, q=0):
    out = []
    for c, x in zip(clk, d):
        if c == (1 if high else 0):
            q = x
        out.append(q)
    return out


def flop(clk, d, q=0):
    out, prev = [], clk[0]
    for c, x in zip(clk, d):
        if c == 1 and prev == 0:
            q = x
        out.append(q)
        prev = c
    return out


clk = [0, 1, 1, 0, 0, 1, 1, 0]
print("Example 08-06.2  latch:", latch(clk, [0, 0, 1, 1, 1, 1, 0, 0]), " flop:", flop(clk, [0, 0, 1, 1, 1, 1, 0, 0]))
print("Problem 08-06.2  pos latch:", latch(clk, [1, 1, 0, 0, 1, 0, 1, 0]), " neg latch:",
      latch(clk, [1, 1, 0, 0, 1, 0, 1, 0], high=False), " flop:", flop(clk, [1, 1, 0, 0, 1, 0, 1, 0]))

rng = np.random.default_rng(806)
clk_long = [(t // 2) % 2 for t in range(4000)]          # 4 samples per clock period
for p_change in (0.0, 0.1, 0.5):
    d = [0]
    for t in range(1, 4000):
        # D may change only while CLK is low (p_change = 0) or also while it is high
        if clk_long[t] == 0 or rng.random() < p_change:
            d.append(int(rng.integers(0, 2)))
        else:
            d.append(d[-1])
    a, b = latch(clk_long, d), flop(clk_long, d)
    print(f"P(change while CLK high) = {p_change}: latch and flop differ in {np.mean(np.array(a) != np.array(b)):.1%} of samples")

Example 08-06.2  latch: [0, 0, 1, 1, 1, 1, 0, 0]  flop: [0, 0, 0, 0, 0, 1, 1, 1]
Problem 08-06.2  pos latch: [0, 1, 0, 0, 0, 0, 1, 1]  neg latch: [1, 1, 1, 0, 1, 1, 1, 0]  flop: [0, 1, 1, 1, 1, 0, 0, 0]
P(change while CLK high) = 0.0: latch and flop differ in 0.0% of samples
P(change while CLK high) = 0.1: latch and flop differ in 3.7% of samples
P(change while CLK high) = 0.5: latch and flop differ in 18.6% of samples


When $D$ never changes while the latch is transparent, the latch passes exactly the value the flip-flop captures at the edge, and in this sampled model the two agree everywhere. The more often $D$ moves while the clock is high, the more often the latch's output departs from the flop's: transparency is only harmless when the data is quiet during the transparent phase, which is exactly what a pipeline of latches on one clock phase cannot guarantee.

## Part 4 · Where a path meets timing

In [5]:
tpcq, tsu, tccq, th = 50, 40, 30, 60
Tc = np.linspace(200, 800, 241)
tpd = np.linspace(0, 700, 281)
TT, PP = np.meshgrid(Tc, tpd)
setup_ok = tpcq + PP + tsu <= TT
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.contourf(TT, PP, setup_ok, levels=[-0.5, 0.5, 1.5], colors=["#F0E442", "#56B4E9"], alpha=0.6)
ax.plot(Tc, Tc - tpcq - tsu, "k-", lw=1)
ax.set_xlabel("clock period $T_c$ (ps)"); ax.set_ylabel("logic delay $t_{pd}$ (ps)")
ax.set_title("Setup met (blue) when $t_{pcq} + t_{pd} + t_{setup} \\le T_c$")
plt.show()
for tcd in (10, 30, 40):
    print(f"t_cd = {tcd} ps: hold slack {tccq + tcd - th:+d} ps at every clock period")

t_cd = 10 ps: hold slack -20 ps at every clock period
t_cd = 30 ps: hold slack +0 ps at every clock period
t_cd = 40 ps: hold slack +10 ps at every clock period


The setup boundary is a line of slope 1: every picosecond of logic needs a picosecond of period. The hold slack is the same at every period: no clock speed fixes it.

## What to take away

1. Two inversions in a loop store a bit; the balance point between the states is unstable, and leaving it takes time $\tau\ln(\Delta V/\Delta V_0)$.
2. A latch is a window, a flip-flop a camera; pipelines need cameras.
3. Setup limits the clock period; hold is independent of it.